# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# The 'metadata' attribute describes the dataset and provides all @ids for referencing entities
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets and their fields. For each entity, we use the `@id` (accessed as `.id`) for referencing in further steps.

`mlcroissant` allows us to introspect the dataset structure as defined in the Croissant schema. We print the list of available record sets and details about their fields and columns.

In [ ]:
# List all record sets: they are accessed via metadata.record_sets
if not hasattr(metadata, 'record_sets') or not metadata.record_sets:
    print("No record sets found in this dataset.")
else:
    print("Record Sets:\n")
    for rs in metadata.record_sets:
        print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
        print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
        if hasattr(rs, 'fields'):
            print("  Fields:")
            for field in rs.fields:
                print(f"    - Field name: {field.name}, @id: {field.id}, type: {getattr(field, 'data_type', '?')}")
        print()

Below is an overview of the first few records from a chosen record set (by `@id`).

Replace `<record_set_id>` with a valid record set `@id` from the previous cell's output.

In [ ]:
# Choose a record set to preview. This dataset only has one main record set.
# We'll programmatically select the first record set if present.
if hasattr(metadata, 'record_sets') and len(metadata.record_sets) > 0:
    chosen_record_set = metadata.record_sets[0]
    chosen_record_set_id = chosen_record_set.id
    print(f"Exploring record set: {chosen_record_set.name} (id: {chosen_record_set_id})\n")
    print("Sample records:")
    for i, record in enumerate(dataset.records(record_set=chosen_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets present in the dataset.")

## 3. Data Extraction
Load all data from the selected record set into a Pandas DataFrame for analysis. All further field and column references use their `@id` attributes (available as `.id` in `mlcroissant`).

In [ ]:
# Extract all available record set @ids
if hasattr(metadata, 'record_sets'):
    record_set_ids = [rs.id for rs in metadata.record_sets]
else:
    record_set_ids = []

dataframes = {}
for record_set_id in record_set_ids:
    # Returns an iterator of records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set {record_set_id}: DataFrame shape {df.shape}")

# For all further analysis we'll use the main record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print("\nAvailable columns (@id) in DataFrame:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head(5)
else:
    print("No record set loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data by key attributes, to prepare for further analysis. All field references use their `@id`.

*In this example, we'll pick a numeric field from the available columns, and perform filtering, normalization, and grouping.*

In [ ]:
# Identify a numeric field for EDA. We'll search for common protein quantification column names (case-insensitive).
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to infer a numeric field (e.g. coverage, mw, peptide counts, normalized abundance)
    candidate_numeric_columns = [
        col for col in df.columns if any(keyword in col.lower() for keyword in ['coverage', 'abundance', 'mw', 'weight', 'count', 'intensity'])
    ]
    print(f"Candidate numeric fields (by @id): {candidate_numeric_columns}")
    if len(candidate_numeric_columns) > 0:
        numeric_field_id = candidate_numeric_columns[0]
        print(f"Selecting field for analysis: {numeric_field_id}")
    else:
        # Fallback: select any float-like column
        float_columns = df.select_dtypes(include=[np.number]).columns.tolist()
        if float_columns:
            numeric_field_id = float_columns[0]
            print(f"Selecting numeric field: {numeric_field_id}")
        else:
            numeric_field_id = df.columns[0]
            print(f"Defaulting to first column: {numeric_field_id}")

    # Ensure this field is numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Remove extreme outliers for demonstration (keep values between 1st and 99th percentile)
    lower = df[numeric_field_id].quantile(0.01)
    upper = df[numeric_field_id].quantile(0.99)
    filtered_df = df[(df[numeric_field_id] > lower) & (df[numeric_field_id] < upper)]
    print(f"Filtered records with {numeric_field_id} in range .01-.99 quantile:")
    print(filtered_df.head())

    # Normalize the numeric column
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Pick a group-by field: select a categorical column (e.g. sample label, accession, or similar)
    candidate_group_columns = [
        col for col in df.columns if any(keyword in col.lower() for keyword in ['sample', 'condition', 'accession'])
    ]
    if candidate_group_columns:
        group_field_id = candidate_group_columns[0]
        print(f"\nGrouping by: {group_field_id}\n")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No categorical field suitable for grouping found.")
else:
    print("No main record set DataFrame loaded.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and, if available, show grouped mean values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if 'group_field_id' in locals():
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to access and analyze a [Croissant FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` Python library. Key steps included:

- Loading dataset metadata and exploring the structure via `@id` references.
- Extracting records programmatically and creating pandas DataFrames.
- Performing filtering and normalization operations on a chosen numeric field.
- Grouping and visualizing the processed data.

The Croissant format allows for rich, machine-readable mapping between schema and tabular/structured datasets, greatly aiding reproducible research and automated data processing pipelines.